# 00 — Charger Bronze (OPTIONNEL)
**À lancer seulement si tes tables `workspace.bronze.*` n'existent pas encore.**
Tu as déjà Bronze ? Ne lance rien ici, passe directement à `01_bronze_checks`.

**Ce que fait ce notebook :** lit les 8 CSV du Volume, garde **toutes les colonnes en texte**, ajoute `_loaded_at`.
Par sécurité, une table qui existe déjà est **ignorée** (sauf si tu mets `OVERWRITE = True`).

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE = f"{CATALOG}.bronze"
VOLUME = "/Volumes/workspace/bronze/raw"   # <- adapte si tes CSV sont ailleurs
OVERWRITE = False                           # True = recharge même si la table existe

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE}")

fichiers = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_name_translation": "product_category_name_translation.csv",
}

for table, fichier in fichiers.items():
    cible = f"{BRONZE}.{table}"
    if spark.catalog.tableExists(cible) and not OVERWRITE:
        print(f"⏭️  {table} existe déjà, ignorée")
        continue
    df = (spark.read.option("header", True)
          .option("multiLine", True)      # les commentaires d'avis contiennent des retours à la ligne
          .option("escape", '"')
          .csv(f"{VOLUME}/{fichier}")     # pas d'inferSchema : tout reste en texte
          .withColumn("_loaded_at", F.current_timestamp()))
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(cible)
    print(f"✅ {table} chargée : {df.count()} lignes")

Ensuite : ouvre `01_bronze_checks` pour vérifier les comptages.